In [ ]:
'''REMAKE EMBEDDINGS AND METRICS FILES FROM OLD EXPERIMENTS'''
import os
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

OLD_EMBEDDINGS_FILENAME = "embeddings.parquet"
OLD_METRICS_FILENAME = "metrics.csv"

MEMORY_FILENAME = "memory.db"
EMBEDDINGS_FILENAMES = {
    "lexical": "embeddings_lexical.parquet",
    "semantic": "embeddings_semantic.parquet",
}
METRICS_CONFIGS = [
    {"filename": "metrics_lexical.csv", "embeddings": "lexical", "role": None, "trim": None},
    {"filename": "metrics_semantic.csv", "embeddings": "semantic", "role": None, "trim": None},
    {"filename": "metrics_lexical_user.csv", "embeddings": "lexical", "role": "user", "trim": None},
    {"filename": "metrics_semantic_user.csv", "embeddings": "semantic", "role": "user", "trim": None},
    {"filename": "metrics_lexical_assistant.csv", "embeddings": "lexical", "role": "assistant", "trim": None},
    {"filename": "metrics_semantic_assistant.csv", "embeddings": "semantic", "role": "assistant", "trim": None},
    {"filename": "metrics_lexical_trim1.csv", "embeddings": "lexical", "role": None, "trim": 1},
    {"filename": "metrics_semantic_trim1.csv", "embeddings": "semantic", "role": None, "trim": 1},
    {"filename": "metrics_lexical_user_trim1.csv", "embeddings": "lexical", "role": "user", "trim": 1},
    {"filename": "metrics_semantic_user_trim1.csv", "embeddings": "semantic", "role": "user", "trim": 1},
    {"filename": "metrics_lexical_assistant_trim1.csv", "embeddings": "lexical", "role": "assistant", "trim": 1},
    {"filename": "metrics_semantic_assistant_trim1.csv", "embeddings": "semantic", "role": "assistant", "trim": 1},
    {"filename": "metrics_lexical_trim2.csv", "embeddings": "lexical", "role": None, "trim": 2},
    {"filename": "metrics_semantic_trim2.csv", "embeddings": "semantic", "role": None, "trim": 2},
    {"filename": "metrics_lexical_user_trim2.csv", "embeddings": "lexical", "role": "user", "trim": 2},
    {"filename": "metrics_semantic_user_trim2.csv", "embeddings": "semantic", "role": "user", "trim": 2},
    {"filename": "metrics_lexical_assistant_trim2.csv", "embeddings": "lexical", "role": "assistant", "trim": 2},
    {"filename": "metrics_semantic_assistant_trim2.csv", "embeddings": "semantic", "role": "assistant", "trim": 2},
]

def remake_embeddings_file(experiment_dir):
    memory_path = experiment_dir / MEMORY_FILENAME
    old_embeddings_path = experiment_dir / OLD_EMBEDDINGS_FILENAME
    new_embeddings_path = {
        "lexical": experiment_dir / EMBEDDINGS_FILENAMES["lexical"],
        "semantic": experiment_dir / EMBEDDINGS_FILENAMES["semantic"],
    }
    with sqlite3.connect(memory_path) as conn:
        are = pd.read_sql_query("SELECT * FROM AttackResultEntries;", conn)
        pme = pd.read_sql_query("SELECT * FROM PromptMemoryEntries;", conn)

    df = pme[pme["conversation_id"].isin(are["conversation_id"])]

    old_emb = pd.read_parquet(old_embeddings_path)

    for name, filename in new_embeddings_path.items():
        emb = df.merge(old_emb[["id", f"embeddings_{name}"]], on="id", how="left")
        emb.rename(columns={f"embeddings_{name}": "embeddings"}, inplace=True)
        emb.to_parquet(filename)

def load_attack_result_entries(experiment_dir):
    memory_path = experiment_dir / MEMORY_FILENAME
    with sqlite3.connect(memory_path) as conn:
        are = pd.read_sql_query("SELECT * FROM AttackResultEntries;", conn)
    are.set_index("conversation_id", inplace=True)
    return are.to_dict(orient="index")

def load_embeddings(experiment_dir):
    return {name: pd.read_parquet(experiment_dir / filename) for name, filename in EMBEDDINGS_FILENAMES.items()}

def get_trajectory(df, conversation_id, role=None, trim=None):
    conversation = df[df["conversation_id"] == conversation_id]
    conversation.sort_values(by=["sequence"], inplace=True)
    conversation.reset_index(drop=True, inplace=True)
    if trim is not None:
        if len(conversation) > 2*trim:
            conversation = conversation.iloc[:-2*trim]
        else:
            return None

    if role is not None:
        conversation = conversation[conversation["role"] == role]
    
    if len(conversation) <= 2:
        return None
    
    traj = np.asarray(conversation["embeddings"].to_list(), dtype=np.float64)
    return traj

def l2_norm_metrics(traj: np.ndarray) -> dict[str, float | int]:
    num_steps = traj.shape[0]

    # 1. Step Distances & Path Length
    steps = np.linalg.norm(traj[1:] - traj[:-1], axis=1)
    distance = np.sum(steps)
    displacement = np.linalg.norm(traj[-1] - traj[0])

    # 2. Circularity (Shape)
    if num_steps < 3:
        circularity = float("nan")
    else:
        centroids = np.mean(traj, axis=0)
        radii = np.linalg.norm(traj - centroids, axis=1)
        circularity = np.std(radii) / (np.mean(radii) + 1e-9)

    # 3. Assemble Metrics
    metrics = {
        "num_steps": int(num_steps),
        "distance": float(distance),
        "displacement": float(displacement),
        "speed": float(np.mean(steps)),
        "speed_std": float(np.std(steps)),
        "velocity": float(displacement / num_steps),
        "directness": float(displacement / (distance + 1e-9)),
        "circularity": float(circularity),
    }

    return metrics

def remake_metrics_files(experiment_dir):
    embeddings = load_embeddings(experiment_dir)
    metrics_configs = [config | {"path": experiment_dir / config["filename"]} for config in METRICS_CONFIGS]
    
    for config in metrics_configs:
        try:
            attack_results = load_attack_result_entries(experiment_dir)
            for conversation_id, result in attack_results.items():
                traj = get_trajectory(
                    embeddings[config["embeddings"]], 
                    conversation_id, 
                    config["role"], 
                    config["trim"])
                if traj is not None:
                    metrics = l2_norm_metrics(traj)
                    result.update(metrics)
            attack_results = pd.DataFrame.from_dict(attack_results, orient="index")
            attack_results.reset_index(inplace=True)
            attack_results.rename(columns={"index": "conversation_id"}, inplace=True)
            attack_results.to_csv(config["path"], index=False)
        except Exception as e:
            print(e)
            continue

run_dir = Path("/Users/muberraozmen/Development/psycho-pass/experiments/objective_model_comparisons")

for experiment_name in os.listdir(run_dir):
    experiment_dir = run_dir / experiment_name
    try:    
        # remake_embeddings_file(experiment_dir)
        remake_metrics_files(experiment_dir)
    except Exception as e:
        print(e)
        continue

In [2]:
''' RECALCULATE EMBEDDINGS AND METRICS FROM SCRATCH '''
from pathlib import Path
import logging
import json
from src.factory import EncoderFactory, MetricsFactory

logger = logging.getLogger(__name__)

experiment_dirs = [
    Path("/Users/muberraozmen/Development/psycho-pass/experiments/objective_model_comparisons/03-29 06:16:41"),
    Path("/Users/muberraozmen/Development/psycho-pass/experiments/objective_model_comparisons/03-29 12:00:49"),
    Path("/Users/muberraozmen/Development/psycho-pass/experiments/objective_model_comparisons/03-29 15:10:47"),
]

for experiment_dir in experiment_dirs:
    try:
        cfg = json.load(open(experiment_dir / "config.json"))
        encoder_factory = EncoderFactory(cfg, experiment_dir, logger)
        encoder_factory.execute()
        metrics_factory = MetricsFactory(cfg, experiment_dir, logger)
        metrics_factory.execute()
    except Exception as e:
        print(e)
        continue
